# Anesthesia AI Data Pipeline — Phase 1
## Predictive Copilot: Data Extraction & Topological Cleaning

In [ ]:
import vitaldb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt

# Nice plots
plt.rcParams['figure.figsize'] = (18, 5)
plt.rcParams['figure.dpi'] = 100

print('All imports OK ✅')

---
## Section 1 — Case Discovery

We need a surgical case that contains:
- **SNUADC/PLETH** — Pulse waveform (plethysmography) at 500 Hz native
- **Primus/CO2** — Capnography waveform at 62.5 Hz native
- A clear **hypotension event** (sudden drop in mean arterial blood pressure)

We'll use `Solar8000/ART_MBP` (numeric mean arterial pressure) to locate the BP drop.

In [ ]:
# Find cases that have BOTH required waveform tracks
required_tracks = ['SNUADC/PLETH', 'Primus/CO2']
candidate_ids = vitaldb.find_cases(required_tracks)
print(f'Found {len(candidate_ids)} cases with PLETH + CO2')

In [ ]:
# Scan candidates for hypotension events (MBP < 65 mmHg)
# We'll check a manageable batch to find a good case quickly

BATCH_SIZE = 50  # check first N candidates
MBP_THRESHOLD = 65  # mmHg — clinical hypotension
MIN_HYPO_DURATION = 30  # seconds — at least 30s of sustained low BP

best_case = None
best_info = {}

for i, cid in enumerate(candidate_ids[:BATCH_SIZE]):
    try:
        # Load MBP at 1 Hz (numeric track, 1-second intervals)
        mbp_data = vitaldb.load_case(cid, ['Solar8000/ART_MBP'], 1)
        if mbp_data is None:
            continue
        mbp = mbp_data[:, 0]

        # Find sustained hypotension windows
        hypo_mask = mbp < MBP_THRESHOLD
        # Count consecutive True values
        valid_mbp = ~np.isnan(mbp)
        hypo_and_valid = hypo_mask & valid_mbp

        if hypo_and_valid.sum() >= MIN_HYPO_DURATION:
            # Find the longest hypotension segment
            hypo_times = np.where(hypo_and_valid)[0]
            # Calculate duration of surgery
            total_minutes = len(mbp) / 60

            # Get the center of the hypotension event
            hypo_center = int(np.median(hypo_times))
            min_mbp = float(np.nanmin(mbp[hypo_times]))

            print(f'  Case {cid}: surgery={total_minutes:.0f}min, '
                  f'hypo_samples={hypo_and_valid.sum()}, '
                  f'min_MBP={min_mbp:.1f}mmHg, '
                  f'event_center={hypo_center}s')

            if best_case is None or hypo_and_valid.sum() > best_info.get('hypo_count', 0):
                best_case = cid
                best_info = {
                    'hypo_count': int(hypo_and_valid.sum()),
                    'hypo_center_s': hypo_center,
                    'min_mbp': min_mbp,
                    'total_min': total_minutes,
                }
    except Exception as e:
        continue

print(f'\n✅ Best case: {best_case}')
print(f'   Details: {best_info}')

---
## Section 2 — Data Extraction at 100 Hz

Extract `SNUADC/PLETH` and `Primus/CO2` at **100 Hz** (interval = 0.01s).
Then slice a **10-15 minute window** centered on the hypotension event.

In [ ]:
# Load the two waveform tracks at 100 Hz
INTERVAL = 1 / 100  # 0.01 seconds — 100 Hz
tracks = ['SNUADC/PLETH', 'Primus/CO2']

raw = vitaldb.load_case(best_case, tracks, INTERVAL)
print(f'Raw data shape: {raw.shape}')
print(f'Duration: {raw.shape[0] * INTERVAL / 60:.1f} minutes')

In [ ]:
# Build time index
n_samples = raw.shape[0]
time = np.arange(n_samples) * INTERVAL

# Create DataFrame
df_full = pd.DataFrame({
    'Time': time,
    'PLETH': raw[:, 0],
    'CO2': raw[:, 1]
})

print(df_full.head())
print(f'\nTotal NaN per column:')
print(df_full.isna().sum())

In [ ]:
# Slice a 12-minute window (720 seconds) centered on the hypotension event
WINDOW_MINUTES = 12
WINDOW_SECONDS = WINDOW_MINUTES * 60
WINDOW_SAMPLES = WINDOW_SECONDS * 100  # at 100 Hz

# Center point in 100Hz samples
center_sample = best_info['hypo_center_s'] * 100

# Calculate start and end, clamped to valid range
start = max(0, center_sample - WINDOW_SAMPLES // 2)
end = min(n_samples, start + WINDOW_SAMPLES)
start = max(0, end - WINDOW_SAMPLES)  # re-adjust if we hit the end

df_window = df_full.iloc[start:end].copy().reset_index(drop=True)

# Re-build clean time column starting from 0
df_window['Time'] = np.arange(len(df_window)) * INTERVAL
df_window['Time'] = df_window['Time'].round(4)

print(f'Window: {len(df_window)} samples = {len(df_window)/100:.1f}s')
print(f'Time range: {df_window["Time"].iloc[0]:.2f}s → {df_window["Time"].iloc[-1]:.2f}s')
print(f'NaN per column:\n{df_window.isna().sum()}')

In [ ]:
# Quick plot of the raw window to verify we captured the event
fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

axes[0].plot(df_window['Time'], df_window['PLETH'], color='#e74c3c', linewidth=0.3, alpha=0.8)
axes[0].set_title('PLETH (Pulse Waveform) — Raw', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_window['Time'], df_window['CO2'], color='#2980b9', linewidth=0.3, alpha=0.8)
axes[1].set_title('CO2 (Capnography) — Raw', fontsize=14, fontweight='bold')
axes[1].set_ylabel('mmHg')
axes[1].set_xlabel('Time (seconds)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('raw_waveforms_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved raw_waveforms_overview.png ✅')

In [ ]:
# Export raw CSV
df_window.to_csv('raw_surgical_data.csv', index=False)
print(f'Exported raw_surgical_data.csv ({len(df_window)} rows) ✅')

---
## Section 3 — NaN Forward-Fill (Zero-Order Hold)

Hospital machines record at slightly different intervals, leaving gaps.
We apply **forward-fill** (copy last valid value into NaN slots), then **back-fill** for any leading NaNs.
The waveform must be mathematically continuous — no nulls allowed.

In [ ]:
df_clean = df_window.copy()

print('NaN BEFORE fill:')
print(df_clean.isna().sum())
print()

# Forward-fill then back-fill
df_clean['PLETH'] = df_clean['PLETH'].ffill().bfill()
df_clean['CO2'] = df_clean['CO2'].ffill().bfill()

print('NaN AFTER fill:')
print(df_clean.isna().sum())

# Assertion — pipeline fails here if any NaN remains
assert df_clean.isna().sum().sum() == 0, 'ERROR: NaN values remain!'
print('\n✅ Zero-order hold complete — no NaN remaining')

---
## Section 4 — Butterworth Low-Pass Filter

Remove high-frequency electrical noise while preserving the shape of heartbeats and breaths.

| Signal | Cutoff | Rationale |
|--------|--------|-----------|
| PLETH  | 20 Hz  | Pulse wave harmonics up to ~15 Hz; keeps waveform shape |
| CO2    | 5 Hz   | Capnography is slower (~0.2-0.5 Hz fundamental); aggressive filtering safe |

We use `sosfiltfilt` for **zero-phase filtering** — no time-shift of peaks.

In [ ]:
def apply_butterworth(signal, cutoff_hz, fs=100, order=4):
    """Apply a zero-phase Butterworth low-pass filter."""
    nyquist = fs / 2
    normalized_cutoff = cutoff_hz / nyquist
    sos = butter(order, normalized_cutoff, btype='low', output='sos')
    return sosfiltfilt(sos, signal)

# Apply filters
df_clean['PLETH_filtered'] = apply_butterworth(df_clean['PLETH'].values, cutoff_hz=20)
df_clean['CO2_filtered'] = apply_butterworth(df_clean['CO2'].values, cutoff_hz=5)

print('Butterworth filtering complete ✅')
print(f'  PLETH: 4th-order low-pass @ 20 Hz')
print(f'  CO2:   4th-order low-pass @ 5 Hz')

---
## Section 5 — Visual Verification

Compare raw vs. filtered on a **zoomed 2-second window**.
Verify that:
1. Heartbeat / breath peaks are in the **exact same time position**
2. High-frequency jaggedness (noise) is **removed**

In [ ]:
# Pick a 2-second window in the middle of our data for verification
mid = len(df_clean) // 2
zoom_start = mid
zoom_end = mid + 200  # 2 seconds at 100 Hz

t = df_clean['Time'].iloc[zoom_start:zoom_end]

fig, axes = plt.subplots(2, 1, figsize=(18, 10))

# PLETH comparison
axes[0].plot(t, df_clean['PLETH'].iloc[zoom_start:zoom_end],
             color='#bdc3c7', linewidth=1.2, alpha=0.8, label='Raw (noisy)')
axes[0].plot(t, df_clean['PLETH_filtered'].iloc[zoom_start:zoom_end],
             color='#e74c3c', linewidth=2, label='Filtered (20 Hz LP)')
axes[0].set_title('PLETH — Raw vs Filtered (2-second zoom)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Amplitude')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# CO2 comparison
axes[1].plot(t, df_clean['CO2'].iloc[zoom_start:zoom_end],
             color='#bdc3c7', linewidth=1.2, alpha=0.8, label='Raw (noisy)')
axes[1].plot(t, df_clean['CO2_filtered'].iloc[zoom_start:zoom_end],
             color='#2980b9', linewidth=2, label='Filtered (5 Hz LP)')
axes[1].set_title('CO2 — Raw vs Filtered (2-second zoom)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('mmHg')
axes[1].set_xlabel('Time (seconds)')
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('filtered_vs_raw_verification.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved filtered_vs_raw_verification.png ✅')

In [ ]:
# Full overview: filtered data across entire window
fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

axes[0].plot(df_clean['Time'], df_clean['PLETH_filtered'], color='#e74c3c', linewidth=0.3)
axes[0].set_title('PLETH — Filtered (Full Window)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_clean['Time'], df_clean['CO2_filtered'], color='#2980b9', linewidth=0.3)
axes[1].set_title('CO2 — Filtered (Full Window)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('mmHg')
axes[1].set_xlabel('Time (seconds)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('filtered_waveforms_full.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved filtered_waveforms_full.png ✅')

---
## Section 6 — Production Export

Final export: only `Time`, `PLETH`, `CO2` (filtered values). No NaN. Strict 0.01s time step.

In [ ]:
# Build production DataFrame with filtered values
df_prod = pd.DataFrame({
    'Time': df_clean['Time'],
    'PLETH': df_clean['PLETH_filtered'],
    'CO2': df_clean['CO2_filtered']
})

# ── Final validation checks ──
print('═' * 50)
print('PRODUCTION DATA VALIDATION')
print('═' * 50)

# 1. No NaN
nan_count = df_prod.isna().sum().sum()
print(f'  NaN count:        {nan_count} {"✅" if nan_count == 0 else "❌"}')
assert nan_count == 0, 'FAIL: NaN values present!'

# 2. Correct columns
expected_cols = ['Time', 'PLETH', 'CO2']
cols_ok = list(df_prod.columns) == expected_cols
print(f'  Columns:          {list(df_prod.columns)} {"✅" if cols_ok else "❌"}')
assert cols_ok, f'FAIL: Expected {expected_cols}'

# 3. Time step consistency
time_diffs = df_prod['Time'].diff().dropna().round(4)
step_ok = (time_diffs == 0.01).all()
print(f'  Time step (0.01): {"consistent" if step_ok else "INCONSISTENT"} {"✅" if step_ok else "❌"}')
assert step_ok, 'FAIL: Time steps are not uniformly 0.01s!'

# 4. Row count
print(f'  Total rows:       {len(df_prod):,}')
print(f'  Duration:         {len(df_prod)/100:.1f} seconds = {len(df_prod)/100/60:.1f} minutes')
print(f'  Time range:       {df_prod["Time"].iloc[0]:.2f}s → {df_prod["Time"].iloc[-1]:.2f}s')

print('═' * 50)
print('ALL CHECKS PASSED ✅')
print('═' * 50)

In [ ]:
# Export production CSV
df_prod.to_csv('production_sim_data.csv', index=False)
print(f'\n🚀 Exported production_sim_data.csv ({len(df_prod):,} rows)')
print(f'   File is ready for the high-speed simulator!')